# Three-flavor fermions on the triangular lattice

A hands-on introduction to the **flavor-conserving SU(3) Hubbard model**. We will build its operators, inspect the eight local occupation states, benchmark a small cluster by exact diagonalization, and optimize a variational wavefunction.

The default is a **six-site teaching cluster** with two fermions of each flavor. It is small enough to run on a CPU. Restart the kernel before changing the lattice or particle numbers: Quantax keeps one global system per Python process. The short optimization below demonstrates the workflow; it is not a converged many-body calculation.

In [ ]:
# CPU is the default for this small example. Set JAX_PLATFORMS=cuda externally
# before launching the kernel to use an allocated GPU instead.
import os
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
from pathlib import Path
import sys, json, csv, time, hashlib, warnings

PROJECT = Path.cwd()
# In the shared student directory, prefer its isolated Quantax checkout.
if (PROJECT / 'quantax' / 'quantax' / '__init__.py').exists():
    REPO = PROJECT / 'quantax'
else:
    candidates = [PROJECT, *PROJECT.parents]
    REPO = next((p for p in candidates if (p/'quantax/__init__.py').exists() and (p/'pyproject.toml').exists()), None)
    if REPO is None:
        raise RuntimeError('Open this notebook from ThreeFlavorTriangular or its Quantax checkout.')
sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sparse
from scipy.sparse.linalg import eigsh
import jax
import jax.numpy as jnp
import fermix
import quantax as qtx
warnings.filterwarnings('ignore', category=fermix.FermixFallbackWarning)
jax.config.update('jax_enable_x64', True)
jax.config.update('jax_default_matmul_precision', 'highest')
qtx.set_default_dtype(jnp.float64)
qtx.set_random_seed(42)
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print('Quantax:', qtx.__file__)
print('Devices:', jax.devices())
assert hasattr(qtx.operator, 'HubbardFlavors'), 'Use the three-flavor checkout for this notebook.'


## 1. Hamiltonian and conventions

With flavor $a\in\{0,1,2\}$ (red, green, blue),
$$H=-t\sum_{\langle ij\rangle,a}(c^\dagger_{ia}c_{ja}+c^\dagger_{ja}c_{ia})
+U\sum_i\sum_{a<b}n_{ia}n_{ib}.$$
Every **unordered spatial bond** is counted once, with its Hermitian conjugate included. Every unordered flavor pair is counted once. There is no flavor conversion, so $[H,N_a]=0$ separately for all three flavors. Equal hoppings and pair repulsions give SU(3) symmetry; fixing $(N_0,N_1,N_2)$ selects a population sector.

Different flavors can occupy the same site. A doubly occupied site costs $U$; a triply occupied site costs **$3U$**, since it contains three flavor pairs. Pauli exclusion forbids two fermions of the same flavor in a single orbital.

With three available modes per site, **one particle per physical site is one-third filling**. Half filling would mean $3/2$ particles per site.

Quantax uses the ordering
$$\underbrace{(0,0),\ldots,(N_s-1,0)}_{\text{flavor 0}},\quad
\underbrace{(0,1),\ldots,(N_s-1,1)}_{\text{flavor 1}},\quad
\underbrace{(0,2),\ldots,(N_s-1,2)}_{\text{flavor 2}}.$$
The configuration has `3*Nphysical` entries, each $+1$ (occupied) or $-1$ (empty). `lattice.Nphysical` is the number used for energy/density **per physical site**. The spinless backend's `Nsites` and `Nmodes` both count all flavor modes.

In [ ]:
EXTENT = (3, 2)
POPULATIONS = (2, 2, 2)
T = 1.0
U = 4.0
RUN_ED = True
STEPS = 30
SAMPLES = 128
C0 = 0.05
ANSATZ = 'slater_jastrow'  # or 'cnn_pfaffian' for an even total particle number

lattice = qtx.sites.TriangularFlavors(EXTENT, Nparticles=POPULATIONS)
print(f'{lattice.Nphysical} physical sites; {lattice.Nmodes} modes; populations {lattice.flavor_counts}')
print('Total particles:', lattice.Ntotal)
print('Particles / physical site:', lattice.Ntotal / lattice.Nphysical)
print('Fraction of all flavor modes occupied:', lattice.Ntotal / lattice.Nmodes)
print('Mode for site 1, flavor 2:', lattice.mode_index(1, 2))

H = qtx.operator.HubbardFlavors(U=U, t=T)
K = qtx.operator.HubbardFlavors(U=0., t=T)

# The same elementary operators can build custom interactions or observables.
c_dagger = qtx.operator.create_flavor(0, 1)
c = qtx.operator.annihilate_flavor(0, 1)
n_green = qtx.operator.number_flavor(0, 1)
bond_red = qtx.operator.hopping_flavor(0, 1, flavor=0, t=T)
onsite_pair = qtx.operator.number_flavor(0, 0) @ qtx.operator.number_flavor(0, 1)

coords = lattice.coord[:lattice.Nphysical]
bonds = lattice.get_neighbor()
physical_bonds = bonds[bonds[:, 1] < lattice.Nphysical]
assert len(bonds) == 3*len(physical_bonds)
assert np.all(bonds[:, 0]//lattice.Nphysical == bonds[:, 1]//lattice.Nphysical)
print('Unordered spatial bonds:', len(physical_bonds))


A very small periodic cluster can identify different infinite-lattice bond directions. Here the Hamiltonian uses the unique unordered pairs returned by the neighbor graph; it does not add extra copies for those coincident directions. For bulk triangular coordination, start with spatial extents of at least three in both directions. The exact-diagonalization section is deliberately limited to small Hilbert spaces.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), layout='constrained')
for i, j in physical_bonds:
    if np.isclose(np.linalg.norm(coords[i]-coords[j]), 1.):
        axes[0].plot(*coords[[i,j]].T, color='0.75', lw=1.5, zorder=1)
axes[0].scatter(*coords.T, s=130, color='#375a7f', edgecolor='white', zorder=2)
axes[0].set(aspect='equal', xlabel='x', ylabel='y', title='Physical triangular cluster (boundary bonds omitted)')
from itertools import product
local_states = np.array(list(product([0,1], repeat=3)))
local_n = local_states.sum(axis=1)
local_cost = U*local_n*(local_n-1)/2
axes[1].bar(range(8), local_cost, color=['0.75' if n < 2 else '#d78b35' for n in local_n])
axes[1].set_xticks(range(8), [''.join(map(str, row)) for row in local_states], rotation=45)
axes[1].set(xlabel='Local occupation (red, green, blue)', ylabel='Interaction energy', title=f'On-site energies at U={U:g}')
plt.show()
plt.close(fig)


## 2. Exact benchmark in a fixed flavor sector

The underlying QuSpin basis fixes the **total** number of spinless modes occupied. We therefore explicitly select the desired flavor populations before diagonalizing. `ParticleHop` uses flavor-preserving neighbor tables, and `rand_states` initializes the requested populations, so the VMC chains below stay in this same sector automatically. Do not use an unrestricted `ExactSampler` as though it already imposed all three populations.

At $U=0$, the ground energy is the sum of occupied one-body energies for each flavor. At $U>0$, exact diagonalization gives a reference for this finite cluster.

In [ ]:
ED_ENERGY = None
ed_rows = []
if RUN_ED:
    from math import comb
    full_dimension = comb(lattice.Nmodes, lattice.Ntotal)
    if full_dimension > 50_000:
        raise ValueError(f'Total-number basis has {full_dimension:,} states. Set RUN_ED=False for larger clusters.')
    identity = qtx.symmetry.Identity()
    identity.basis_make()
    all_spins = qtx.utils.ints_to_array(identity.basis.states)
    all_counts = (all_spins.reshape(-1, lattice.nflavors, lattice.Nphysical)>0).sum(axis=-1)
    sector = np.flatnonzero(np.all(all_counts == np.array(POPULATIONS), axis=1))
    sector_spins = all_spins[sector]
    print(f'Total-number basis: {full_dimension:,}; selected flavor sector: {len(sector):,}')
    kinetic = K.get_quspin_op().tocsr()[sector][:, sector]
    occ = (sector_spins.reshape(-1, 3, lattice.Nphysical)>0).astype(float)
    density = occ.sum(axis=1)
    pair_count = (density*(density-1)/2).sum(axis=1)
    triple_count = np.prod(occ, axis=1).sum(axis=1)
    rng = np.random.default_rng(82)
    eigenvectors = []
    for interaction in sorted(set([0., 2., U, 8.])):
        matrix = kinetic + sparse.diags(interaction*pair_count)
        values, vectors = eigsh(matrix, k=1, which='SA', tol=1e-10, v0=rng.normal(size=len(sector)))
        psi = vectors[:,0]
        probability = np.abs(psi)**2
        residual = np.linalg.norm(matrix@psi-values[0]*psi)
        assert residual < 1e-7
        ed_rows.append(dict(U=interaction, energy_per_site=float(values[0]/lattice.Nphysical),
                            pairs_per_site=float(probability@pair_count/lattice.Nphysical),
                            triples_per_site=float(probability@triple_count/lattice.Nphysical)))
        eigenvectors.append(psi)
        if interaction == U:
            ED_ENERGY = float(values[0])
            H_sector = matrix
    one_body = np.zeros((lattice.Nphysical, lattice.Nphysical))
    for i,j in physical_bonds:
        one_body[i,j] -= T
        one_body[j,i] -= T
    free_energy = sum(np.linalg.eigvalsh(one_body)[:count].sum() for count in POPULATIONS)
    np.testing.assert_allclose(ed_rows[0]['energy_per_site']*lattice.Nphysical, free_energy, atol=1e-8)
    print('Checked U=0 against independent one-body filling.')
    print('U/t       exact E/site       pairs/site       triples/site')
    for row in ed_rows:
        print(f"{row['U']:4g}      {row['energy_per_site']: .8f}       {row['pairs_per_site']:.6f}          {row['triples_per_site']:.6f}")


In [ ]:
if ed_rows:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), layout='constrained')
    axes[0].plot([r['U'] for r in ed_rows], [r['energy_per_site'] for r in ed_rows], 'o-', color='C0')
    axes[0].set(xlabel='U/t', ylabel='Ground energy / physical site')
    axes[1].plot([r['U'] for r in ed_rows], [r['pairs_per_site'] for r in ed_rows], 'o-', label='Flavor pairs')
    axes[1].plot([r['U'] for r in ed_rows], [r['triples_per_site'] for r in ed_rows], 's--', label='Triple occupancy')
    axes[1].set(xlabel='U/t', ylabel='Mean number / physical site')
    axes[1].legend(frameon=False)
    for ax in axes:
        ax.grid(alpha=.2)
    plt.show()
    plt.close(fig)


## 3. A small variational calculation

The default ansatz is a determinant multiplied by a trainable density Jastrow factor. It works for either odd or even total particle number. We initialize the determinant from the lowest one-body spatial orbitals (with a small real perturbation to lift finite-cluster degeneracies) copied into separate flavor blocks; the Jastrow factor adds density correlations. The variational state is sampled in the chosen flavor sector.

For a more expressive neural option, set `ANSATZ='cnn_pfaffian'`. This uses Quantax's residual CNN and multiplicative pairing head, with one set of input channels for each flavor. A standard Pfaffian needs an **even** total particle number; for an odd number, use the determinant example or implement an explicit blocked-orbital Pfaffian.

We use norm-constrained AdamSR with a small $C_0$. This caps the complete parameter update; it is not an additional scalar multiplier applied afterward. All three populations are checked during sampling.

In [ ]:
if ANSATZ == 'slater_jastrow':
    rng = np.random.default_rng(42)
    one_body = np.zeros((lattice.Nphysical, lattice.Nphysical))
    for i,j in physical_bonds:
        one_body[i,j] -= T
        one_body[j,i] -= T
    # Lift finite-cluster degeneracies gently to choose generic nonzero minors.
    perturbation = rng.normal(size=one_body.shape)
    _, spatial_orbitals = np.linalg.eigh(one_body + .01*(perturbation+perturbation.T))
    orbitals = np.zeros((lattice.Nmodes, lattice.Ntotal))
    column = 0
    for flavor, count in enumerate(POPULATIONS):
        lo = flavor*lattice.Nphysical
        orbitals[lo:lo+lattice.Nphysical, column:column+count] = spatial_orbitals[:, :count]
        column += count
    mean_field = qtx.model.GeneralDet(U=jnp.asarray(orbitals), dtype=jnp.float64)
    model = qtx.model.GeneralJastrow(qtx.model.Jastrow(dtype=jnp.float64), mean_field)
elif ANSATZ == 'cnn_pfaffian':
    if lattice.Ntotal % 2:
        raise ValueError('The unblocked Pfaffian needs an even total particle number.')
    model = qtx.model.MultiplicativePfBackflow.resconv(d=4, channels=8, nblocks=1,
                                                      dtype=jnp.float64, pairing_dtype=jnp.float64)
else:
    raise ValueError('Choose slater_jastrow or cnn_pfaffian')
state = qtx.state.Variational(model, max_parallel=(256, 32, 256))
sampler = qtx.sampler.ParticleHop(state, SAMPLES, thermal_steps=20*lattice.Nphysical,
                                sweep_steps=4*lattice.Nphysical)
optimizer = qtx.optimizer.AdamSR(state, H, C0=C0, t0=STEPS, T=1000,
                                clip_mode='median_mad')
np.testing.assert_array_equal(np.asarray(lattice.count_flavors(sampler.sweep(0).spins)),
                              np.tile(POPULATIONS, (SAMPLES,1)))
print('Ansatz:', ANSATZ, '; trainable parameters:', state.nparams)
print('Each chain has populations', POPULATIONS)


Outputs go to a new timestamped directory on each run. Checkpoints retain the model, optimizer, chain endpoints, RNG and history together; final evaluation retains each original sample block and its local energies separately.

In [ ]:
run_dir = PROJECT / 'runs' / f'three_flavor_{time.time_ns()}'
run_dir.mkdir(parents=True)
config = dict(extent=list(EXTENT), populations=list(POPULATIONS), physical_sites=lattice.Nphysical,
              modes=lattice.Nmodes, U=U, t=T, ansatz=ANSATZ, C0=C0, steps=STEPS, samples=SAMPLES,
              quantax_path=str(REPO), dtype='float64')
for name in ['sites/flavor_lattice.py', 'operator/flavor_operators.py', 'utils/basis.py']:
    config[name+'_sha256'] = hashlib.sha256((REPO/'quantax'/name).read_bytes()).hexdigest()
(run_dir/'config.json').write_text(json.dumps(config, indent=2)+'\n')
if RUN_ED:
    np.savez_compressed(run_dir/'exact_diagonalization.npz', sector_spins=sector_spins,
                        eigenvectors=np.asarray(eigenvectors), **{key:np.array([r[key] for r in ed_rows]) for key in ed_rows[0]})
# This helper commits the complete model/optimizer/sampler/RNG boundary atomically.
sys.path.insert(0, str(REPO/'examples/organics/source'))
from training_resume import save_boundary
history = []
started = time.monotonic()
save_boundary(run_dir, state, optimizer, sampler, 0, 0.)
with (run_dir/'history.csv').open('w', newline='', buffering=1) as stream:
    fields = ['iteration','energy_per_site','variance','step_norm']
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    for step in range(1, STEPS+1):
        samples = sampler.sweep()
        np.testing.assert_array_equal(np.asarray(lattice.count_flavors(samples.spins)),np.tile(POPULATIONS,(SAMPLES,1)))
        update = optimizer.get_step(samples)
        energy, variance = float(optimizer.energy), float(optimizer.VarE)
        norm = float(jnp.linalg.norm(update))
        if not np.isfinite([energy,variance,norm]).all():
            raise FloatingPointError('Nonfinite training values; inspect the saved checkpoint.')
        assert norm <= C0*(1+1e-6)
        state.update(update)
        jax.block_until_ready(state.get_params_flatten())
        row = dict(iteration=step,energy_per_site=energy/lattice.Nphysical,variance=variance,step_norm=norm)
        writer.writerow(row)
        history.append(row)
        if step%10 == 0 or step == STEPS:
            stream.flush()
            os.fsync(stream.fileno())
            save_boundary(run_dir,state,optimizer,sampler,step,time.monotonic()-started)
            print(f"Step {step:3d}: training E/site = {row['energy_per_site']:.6f}; norm = {norm:.4f}")
print('Saved to', run_dir)


## 4. Frozen-state evaluation

Training energies above belong to the parameters **before** their update. We now hold the final weights fixed, warm the chains again, and save four original sample blocks. The error bar is the SEM of their four energy means; it does not include optimization bias or residual autocorrelation. A short run need not reach the exact ground state.

For this small cluster we also compute the exact Rayleigh quotient of the trained trial wavefunction in the same flavor sector. Unlike a noisy Monte Carlo estimate, that quotient must lie above the exact ground energy.

In [ ]:
state.save(run_dir/'final_model.eqx')
optimizer.save(run_dir/'final_optimizer.eqx')
sampler.sweep(40*lattice.Nphysical)
block_energies = []
occupations = []
for block in range(4):
    samples = sampler.sweep()
    spins = np.asarray(samples.spins)
    local = np.asarray(H.Oloc(state, samples)).real
    if not np.isfinite(local).all():
        raise FloatingPointError('Nonfinite frozen-state local energies')
    np.savez_compressed(run_dir/f'evaluation_{block+1:03d}.npz', spins=spins, energy_locals=local,
                        chain_index=np.arange(SAMPLES))
    block_energies.append(float(local.mean()/lattice.Nphysical))
    occupations.append((spins.reshape(-1,3,lattice.Nphysical)>0).astype(float))
energy_mean = float(np.mean(block_energies))
energy_sem = float(np.std(block_energies,ddof=1)/np.sqrt(len(block_energies)))
print(f'Frozen Monte Carlo energy/site: {energy_mean:.7f} ± {energy_sem:.7f} (original-block SEM)')
trial_exact = None
if RUN_ED:
    amplitudes = np.asarray(state(jnp.asarray(sector_spins)))
    norm = np.vdot(amplitudes,amplitudes).real
    trial_exact = float(np.vdot(amplitudes,H_sector@amplitudes).real/norm/lattice.Nphysical)
    assert trial_exact >= ED_ENERGY/lattice.Nphysical-1e-8
    print(f'Exact trial-state Rayleigh quotient/site: {trial_exact:.7f}')
    print(f'Exact ground energy/site:                {ED_ENERGY/lattice.Nphysical:.7f}')
summary = dict(block_energy_per_site=block_energies, energy_per_site=energy_mean,
               block_sem=energy_sem, exact_trial_energy_per_site=trial_exact,
               ground_energy_per_site=None if ED_ENERGY is None else ED_ENERGY/lattice.Nphysical,
               total_evaluation_samples=4*SAMPLES)
(run_dir/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4),layout='constrained')
axes[0].plot([r['iteration'] for r in history],[r['energy_per_site'] for r in history],color='C0',alpha=.7,label='Training estimates')
axes[0].errorbar(STEPS+1,energy_mean,yerr=energy_sem,fmt='o',color='C1',capsize=4,label='Frozen sample estimate')
if ED_ENERGY is not None:
    axes[0].axhline(ED_ENERGY/lattice.Nphysical,color='black',ls='--',label='Exact ground state')
if trial_exact is not None:
    axes[0].axhline(trial_exact,color='C2',ls=':',label='Exact trial-state energy')
axes[0].set(xlabel='Optimization update',ylabel='Energy / physical site')
axes[0].legend(frameon=False,fontsize=8)
axes[0].grid(alpha=.2)
mean_occupation=np.concatenate(occupations).mean(axis=0)
im=axes[1].imshow(mean_occupation,vmin=0,vmax=1,cmap='viridis',aspect='auto')
axes[1].set_yticks(range(3),['red (0)','green (1)','blue (2)'])
axes[1].set(xlabel='Physical site index',ylabel='Flavor',title='Final sampled flavor occupations')
fig.colorbar(im,ax=axes[1],label='Mean occupancy')
fig.savefig(run_dir/'energy_and_occupations.png',dpi=180)
plt.show()
plt.close(fig)


## Suggested next experiments

1. Increase the number of updates and compare seeds at fixed $U$. Inspect the exact trial-state energy and the frozen block estimates, rather than the lowest noisy training point.
2. Vary $U/t$, then compare pair and triple occupancy with the exact small-cluster benchmark.
3. Use `EXTENT=(3,3)` with populations `(3,3,3)` for nine physical sites at one particle per site. Set `RUN_ED=False`; use the determinant ansatz for its odd total particle number. Restart the kernel first.
4. Try unequal populations to study flavor polarization. The hopping remains flavor conserving; population imbalance selects a different sector.
5. Compare the determinant–Jastrow and CNN–Pfaffian ansätze on an even-particle cluster. Their representational limitations differ; energy differences are not by themselves evidence for different physical phases.